In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import time
from get_light_status import TrafficLightDetector

In [ ]:
model = YOLO("yolo26m.engine", task="detect")
traffic_light_detector = TrafficLightDetector()

In [ ]:
# Create a blank black image: (height, width, channels)
# 640x640 is standard, dtype must be uint8 for images
warmup_frame = np.zeros((640, 640, 3), dtype=np.uint8)

# Run the warmup
model(warmup_frame)

Loading yolo26m.engine for TensorRT inference...
[03/10/2026-19:18:52] [TRT] [I] Loaded engine size: 42 MiB
[03/10/2026-19:18:52] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +32, now: CPU 0, GPU 71 (MiB)

0: 640x640 (no detections), 2.2ms
Speed: 1.4ms preprocess, 2.2ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 640)


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted p

In [ ]:
OUTPUT_WIDTH = 1280
OUTPUT_HEIGHT = 720

In [ ]:
# # 2. Define Color Ranges (Hue, Saturation, Value)
# # Red has two ranges because it wraps around the 0/180 degree mark
# lower_red1 = np.array([0, 100, 100])
# upper_red1 = np.array([10, 255, 255])
# lower_red2 = np.array([160, 100, 100])
# upper_red2 = np.array([180, 255, 255])

# lower_yellow = np.array([15, 100, 100])
# upper_yellow = np.array([35, 255, 255])

# lower_green = np.array([40, 100, 100])
# upper_green = np.array([90, 255, 255])

# # Configuration
# CLASS_NAMES = ['Green', 'Red', 'Yellow'] # Ensure this matches your training order
# COLOR_MAP = {'Green': (0, 255, 0), 'Red': (0, 0, 255), 'Yellow': (0, 255, 255)}

In [ ]:
# def get_light_state(crop):
#     # 1. Convert to HSV
#     hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
#     h, w, _ = hsv.shape
    

#     # 3. Create Masks
#     mask_red = cv2.add(cv2.inRange(hsv, lower_red1, upper_red1), 
#                     cv2.inRange(hsv, lower_red2, upper_red2))
 
#     mask_yellow = cv2.inRange(hsv, lower_yellow, upper_yellow)
#     mask_green = cv2.inRange(hsv, lower_green, upper_green)

#     # 2. Count non-zero pixels for each color
#     counts = {
#         "Red": cv2.countNonZero(mask_red),
#         "Yellow": cv2.countNonZero(mask_yellow),
#         "Green": cv2.countNonZero(mask_green)
#     }

#     # 3. Find the dominant color
#     state = max(counts, key=counts.get)

#     # 4. Validation: Only return if the dominant color has enough "presence"
#     # A threshold of 10 pixels is very low—consider 5% of total area for robustness
#     if counts[state] < (h * w * 0.05): 
#         return "UNKNOWN"

#     # # 5. Spatial Verification (Optional but recommended)
#     # # Check if the Red is actually in the upper half, etc.
#     # if state == "Red":
#     #     # Does at least 60% of red pixels appear in the top 60% of the box?
#     #     top_half_red = cv2.countNonZero(mask_red[0:int(h*0.6), :])
#     #     if top_half_red < (counts["Red"] * 0.6):
#     #         return "UNKNOWN" # Likely a reflection or tail light, not the signal head

#     return state

#     # # 4. Divide crop into 3 horizontal zones (Top, Middle, Bottom)
#     # height, _ = mask_red.shape
#     # top = mask_red[0:height//3, :]
#     # mid = mask_yellow[height//3:2*height//3, :]
#     # bot = mask_green[2*height//3:height, :]

#     # # 5. Determine state based on pixel density in specific zones
#     # if cv2.countNonZero(top) > 10: # Threshold of 10 pixels
#     #     return "Red"
#     # elif cv2.countNonZero(mid) > 10:
#     #     return "Yellow"
#     # elif cv2.countNonZero(bot) > 10:
#     #     return "Green"
    
#     # return "UNKNOWN"

In [ ]:
print(model.names[9])

traffic light


In [ ]:
# 2. Open Camera (0 is usually the default webcam)
cap = cv2.VideoCapture('video/hongkong2.mp4')

# Optional: Set resolution to match your export size for speed
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 640)

while cap.isOpened():
    # start_time = time.perf_counter()
    success, frame = cap.read()
    if not success:
        break

    # 3. model.track() handles detection + temporal association
    # persist=True tells the tracker to remember IDs from the previous frame
    results = model(frame, classes=[9], verbose=False, stream=True)

    # 2. Iterate through the generator
    for r in results:
        # Access the boxes (standard detections)
        boxes = r.boxes 
        
        # Example: Drawing boxes manually since we removed tracking
        for box in boxes:
            # get coordinates
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)

                # 2. Boundary-safe Cropping
            # Ensure coordinates are within frame limits
            crop = frame[max(0, y1):y2, max(0, x1):x2]
            
            if crop.size == 0:
                continue

            # 3. Direct Classification (No resizing unless get_light_state requires it)
            # Tip: Move cv2.resize INSIDE get_light_state only if absolutely necessary
            # color_label = get_light_state(crop)
            state = traffic_light_detector.get_light_state(crop)
            
            # 4. Immediate Drawing (Skips the list/enumerate overhead)
            # bgr_color = COLOR_MAP.get(color_label, (255, 255, 255))
            bgr_color = traffic_light_detector.COLOR_MAP.get(state, (255, 255, 255)) # Default to White if UNKNOWN
            cv2.rectangle(frame, (x1, y1), (x2, y2), bgr_color, 2)
            
            
            # cv2.putText(frame, f"Light {conf:.2f}", (x1, y1 - 10), 
            #             cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    # 4. Display the output
    display_frame = cv2.resize(frame, (OUTPUT_WIDTH, OUTPUT_HEIGHT), interpolation=cv2.INTER_AREA)
    cv2.imshow("YOLOv11 Traffic Light Tracking", display_frame)

    # end_time = time.perf_counter()
    # print(f"Execution time: {end_time - start_time:.4f} seconds")

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()